# Supplementary Figure 6 | Concordance of GTEx v10 and v11 QTL significance by biotype

**a.** Per-gene eGene −log10q in v10 versus v11, stratified and colored by v11 biotype (protein-coding, lncRNA, pseudogene). Each point is one gene annotated in both GENCODE releases. The minimum q across tissues for that gene is plotted. Untested genes are assigned q = 1 and plotted at NT (not tested; typically failed QTL QC filters); −log10 q values above 10 are clipped. Dashed lines mark q = 0.05 and q = 0.01; shaded areas are those with strong discordance in eGene calls (q < 0.01 in one version and q ≥ 0.05 in the other). Panel titles report Lin’s concordance correlation coefficient (CCC) on −log10 q and the number of strongly discordant genes. Concordance is high overall for protein-coding genes (CCC = 0.996), but 52 are strongly discordant. Many pseudogenes and lncRNAs are strongly discordant.

**b.** Same as (a) for sGenes (minimum q across splicing phenotypes and tissues is plotted). Concordance is high overall for protein-coding genes (CCC = 0.996), but 51 are strongly discordant. lncRNA sGene concordance is high for the biotype overall (CCC=0.911), but 386 genes are strongly discordant. Pseudogene sGene concordance is lower (CCC=0.368).

**c.** For each strongly discordant eGene (circles) or sGene (squares) from (a-b), the change in percent ambiguous exon base pairs (v47 − v39, % of total exon bp in the version that are ambiguous) versus the exon bp unique to one release (plotted as % of v47 exon bp). Strongly discordant genes of all biotypes generally show either annotation change, ambiguity change, or both.


## Setup


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from tqdm import tqdm
from matplotlib.patches import Rectangle

plt.rc("axes", labelsize=6)
plt.rc("xtick", labelsize=6)
plt.rc("ytick", labelsize=6)
plt.rc("legend", fontsize=5)
plt.rc("legend", title_fontsize=6)
plt.rc("font", size=6)
plt.rc("figure", titlesize=7)
plt.rcParams["pdf.fonttype"] = 42
plt.rcParams["ps.fonttype"] = 42
plt.rcParams["figure.figsize"] = (7.2, 7.2)
plt.rcParams["figure.dpi"] = 300
plt.rcParams["savefig.dpi"] = 300

fig_width, fig_height, fig_dpi = 7.2, 7.2, 300

fdr_threshold = 0.05
missing_qval = 1.0  # assigned when gene annotated in both but untested in one release
fdr_strong = 0.01  # strong-flip threshold for labeling
display_max = 10.0  # clip −log10 q for plotting
biotype_order = ["protein_coding", "lncRNA", "pseudogene"]

# GENCODE gene_type → panel biotype (applied to v47 / v11)
protein_coding_types = {
    "protein_coding", "IG_C_gene", "IG_D_gene", "IG_J_gene", "IG_V_gene",
    "TR_C_gene", "TR_D_gene", "TR_J_gene", "TR_V_gene",
}
pseudogene_types = {
    "Mt_tRNA_pseudogene", "tRNA_pseudogene", "snoRNA_pseudogene", "snRNA_pseudogene",
    "scRNA_pseudogene", "noncoding_RNA_pseudogene", "misc_RNA_pseudogene", "miRNA_pseudogene",
    "transcribed_processed_pseudogene", "transcribed_unprocessed_pseudogene",
    "transcribed_unitary_pseudogene", "translated_processed_pseudogene",
    "translated_unprocessed_pseudogene", "unitary_pseudogene", "unprocessed_pseudogene",
    "pseudogene", "polymorphic_pseudogene", "processed_pseudogene",
    "IG_C_pseudogene", "IG_J_pseudogene", "IG_V_pseudogene", "IG_pseudogene",
    "TR_V_pseudogene", "TR_J_pseudogene",
}
lncrna_types = {
    "3prime_overlapping_ncRNA", "antisense", "bidirectional_promoter_lncRNA", "lincRNA",
    "macro_lncRNA", "non_coding", "processed_transcript", "sense_intronic",
    "sense_overlapping", "TEC", "lncRNA",
}


def group_gene_type(t):
    if pd.isna(t):
        return None
    t = str(t)
    if t in protein_coding_types:
        return "protein_coding"
    if t in pseudogene_types:
        return "pseudogene"
    if t in lncrna_types:
        return "lncRNA"
    return None


biotype_labels = {
    "protein_coding": "protein-coding",
    "lncRNA": "lncRNA",
    "pseudogene": "pseudogene",
}
_paired = sns.color_palette("Paired", 12)
gene_type_palette = {
    "protein_coding": _paired[1],
    "lncRNA": _paired[5],
    "pseudogene": _paired[3],
}
# e/sGene file naming inside the QTL summary directories (portal-style releases)
qtl_summary_suffix = {
    "eqtl": {"v10": "v10.eGenes", "v11": "v11.eGenes"},
    "sqtl": {"v10": "v10.sGenes", "v11": "v11.sGenes"},
}


## Paths

In [ ]:
gencode_class_file = Path("path/to/gencode_v39_v47_gene_classification.txt")

qtl_tissue_ids_file = Path("path/to/qtl_tissue_ids.txt")
v10_eqtl_dir = Path("path/to/v10_eqtl_susie")
v10_sqtl_dir = Path("path/to/v10_sqtl_susie")
v11_eqtl_dir = Path("path/to/v11_eqtl_susie")
v11_sqtl_dir = Path("path/to/v11_sqtl_susie")
qtl_summary_dirs = {
    ("v10", "eqtl"): v10_eqtl_dir,
    ("v10", "sqtl"): v10_sqtl_dir,
    ("v11", "eqtl"): v11_eqtl_dir,
    ("v11", "sqtl"): v11_sqtl_dir,
}


fig_dir = Path("figures")
fig_pdf = fig_dir / "supp_figure_6_v10_v11_pc_qtl_concordance.pdf"
fig_png = fig_dir / "supp_figure_6_v10_v11_pc_qtl_concordance.png"


## Load data


### GENCODE classification (both-annotated universe)


In [ ]:
gencode = pd.read_table(gencode_class_file, low_memory=False)
# both-annotated; panel biotype = v11 / GENCODE v47
both_ann = (
    gencode.dropna(subset=["gene_id_stable", "gene_id_v39", "gene_id_v47"])
    .drop_duplicates("gene_id_stable")
    .copy()
)
both_ann["biotype_v47"] = both_ann["gene_type_v47"].map(group_gene_type)
both_ann = both_ann[both_ann["biotype_v47"].isin(biotype_order)].copy()

stable_to_biotype = both_ann.set_index("gene_id_stable")["biotype_v47"]
both_annotated_genes = set(stable_to_biotype.index)

v39_map = (
    gencode.dropna(subset=["gene_id_v39"])
    .drop_duplicates("gene_id_v39")
    .set_index("gene_id_v39")["gene_id_stable"]
)
v47_map = (
    gencode.dropna(subset=["gene_id_v47"])
    .drop_duplicates("gene_id_v47")
    .set_index("gene_id_v47")["gene_id_stable"]
)
id_to_stable = {"v10": v39_map, "v11": v47_map}

print("both-annotated genes by v47 (v11) biotype:")
print(stable_to_biotype.value_counts().reindex(biotype_order).to_string())


### QTL tissue list


In [ ]:
qtl_tissues = pd.read_table(qtl_tissue_ids_file, header=None)[0].tolist()
print(f"Tissues: {len(qtl_tissues)}")


### Compute — per-gene eQTL q-values


In [ ]:
def load_tissue_qvals(tissue, version, qtl_type):
    suffix = qtl_summary_suffix[qtl_type][version]
    path = qtl_summary_dirs[(version, qtl_type)] / f"{tissue}.{suffix}.txt.gz"
    if not path.is_file():
        raise FileNotFoundError(f"Missing QTL summary: {path}")
    # Read only needed columns; fall back if gene_id is absent.
    try:
        df = pd.read_csv(path, sep="\t", usecols=["gene_id", "qval"])
    except ValueError:
        df = pd.read_csv(path, sep="\t", usecols=["phenotype_id", "qval"])
        df["gene_id"] = df["phenotype_id"].str.split(":").str[-1]
    df = df[["gene_id", "qval"]].dropna(subset=["qval"])
    df["gene_id_stable"] = df["gene_id"].map(id_to_stable[version])
    df = df[df["gene_id_stable"].notna()]
    df = df[df["gene_id_stable"].isin(both_annotated_genes)]
    return df.groupby("gene_id_stable", observed=True)["qval"].min()


def pool_gene_qvals(qtl_type):
    """Min q across tissues per version; outer-join onto both-annotated genes; fill untested with missing_qval."""
    q10_parts, q11_parts = [], []
    for tissue in tqdm(qtl_tissues, desc=f"pool {qtl_type}"):
        q10_parts.append(load_tissue_qvals(tissue, "v10", qtl_type))
        q11_parts.append(load_tissue_qvals(tissue, "v11", qtl_type))
    q10 = pd.concat(q10_parts).groupby(level=0).min().rename("qval_v10")
    q11 = pd.concat(q11_parts).groupby(level=0).min().rename("qval_v11")

    out = (
        pd.DataFrame({"gene_id_stable": sorted(both_annotated_genes)})
        .assign(biotype=lambda d: d["gene_id_stable"].map(stable_to_biotype))
        .merge(q10, left_on="gene_id_stable", right_index=True, how="left")
        .merge(q11, left_on="gene_id_stable", right_index=True, how="left")
    )
    out["tested_v10"] = out["qval_v10"].notna()
    out["tested_v11"] = out["qval_v11"].notna()
    out["qval_v10"] = out["qval_v10"].fillna(missing_qval)
    out["qval_v11"] = out["qval_v11"].fillna(missing_qval)
    return out

eqtl_df = pool_gene_qvals("eqtl")


### Per-gene eQTL q-values (minimum across tissues, v10 and v11)

In [ ]:
print(
    f"eQTL genes: {len(eqtl_df):,} | "
    f"tested both={(eqtl_df['tested_v10'] & eqtl_df['tested_v11']).sum():,} | "
    f"v10-only={(eqtl_df['tested_v10'] & ~eqtl_df['tested_v11']).sum():,} | "
    f"v11-only={(eqtl_df['tested_v11'] & ~eqtl_df['tested_v10']).sum():,} | "
    f"neither={(~eqtl_df['tested_v10'] & ~eqtl_df['tested_v11']).sum():,}"
)


### Compute — per-gene sQTL q-values


In [ ]:
# Reuses load_tissue_qvals / pool_gene_qvals from the eQTL build cell above.
sqtl_df = pool_gene_qvals("sqtl")


### Per-gene sQTL q-values (minimum across splicing phenotypes and tissues, v10 and v11)

In [ ]:
print(
    f"sQTL genes: {len(sqtl_df):,} | "
    f"tested both={(sqtl_df['tested_v10'] & sqtl_df['tested_v11']).sum():,} | "
    f"v10-only={(sqtl_df['tested_v10'] & ~sqtl_df['tested_v11']).sum():,} | "
    f"v11-only={(sqtl_df['tested_v11'] & ~sqtl_df['tested_v10']).sum():,} | "
    f"neither={(~sqtl_df['tested_v10'] & ~sqtl_df['tested_v11']).sum():,}"
)


## Run analysis


### Per-gene −log10 q-values and CCC by biotype


In [ ]:
def neglog10_q(q):
    return -np.log10(q.clip(lower=1e-300))


def lin_ccc(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    ok = np.isfinite(x) & np.isfinite(y)
    x, y = x[ok], y[ok]
    if len(x) < 2:
        return np.nan
    mx, my = x.mean(), y.mean()
    vx = ((x - mx) ** 2).mean()
    vy = ((y - my) ** 2).mean()
    cov = ((x - mx) * (y - my)).mean()
    denom = vx + vy + (mx - my) ** 2
    return float(2 * cov / denom) if denom > 0 else np.nan


annot_by_gene = both_ann.set_index("gene_id_stable")[
    ["pct_ambiguous_v39", "pct_ambiguous_v47", "pct_exon_turnover"]
]

pooled_calls = {}
pooled_ccc = {}

for qtl_type, df in (("eqtl", eqtl_df), ("sqtl", sqtl_df)):
    gene_df = df.copy()
    gene_df["biotype"] = gene_df["gene_id_stable"].map(stable_to_biotype)
    gene_df = gene_df[gene_df["biotype"].isin(biotype_order)].copy()
    gene_df = gene_df.join(annot_by_gene, on="gene_id_stable")
    gene_df = gene_df.assign(
        neglog10_q_v10=neglog10_q(gene_df["qval_v10"]),
        neglog10_q_v11=neglog10_q(gene_df["qval_v11"]),
        sig_v10=gene_df["qval_v10"] < fdr_threshold,
        sig_v11=gene_df["qval_v11"] < fdr_threshold,
        strong_v10=gene_df["qval_v10"] < fdr_strong,
        strong_v11=gene_df["qval_v11"] < fdr_strong,
        delta_ambig=(gene_df["pct_ambiguous_v47"] - gene_df["pct_ambiguous_v39"]) * 100.0,
    )
    gene_df["strong_flip"] = (
        (gene_df["strong_v10"] & ~gene_df["sig_v11"])
        | (gene_df["strong_v11"] & ~gene_df["sig_v10"])
    )

    for biotype in biotype_order:
        sub = gene_df[gene_df["biotype"] == biotype].copy()
        pooled_calls[(qtl_type, biotype)] = sub
        pooled_ccc[(qtl_type, biotype)] = lin_ccc(sub["neglog10_q_v10"], sub["neglog10_q_v11"])
        n_v10 = int((sub["strong_v10"] & ~sub["sig_v11"]).sum())
        n_v11 = int((sub["strong_v11"] & ~sub["sig_v10"]).sum())
        n_fill = int((~sub["tested_v10"] | ~sub["tested_v11"]).sum())
        print(
            f"{qtl_type} {biotype}: {len(sub):,} genes "
            f"({n_fill:,} with ≥1 untested→q={missing_qval:g}), "
            f"strong flips v10-only={n_v10} v11-only={n_v11}, "
            f"CCC={pooled_ccc[(qtl_type, biotype)]:.3f}"
        )

### Compute — Storey π₁ replication (v10 ↔ v11)


In [ ]:
def load_tissue_pval_qval(tissue, version, qtl_type):
    """gene_id_stable → qval, pval_beta (best phenotype per gene if needed)."""
    suffix = qtl_summary_suffix[qtl_type][version]
    path = qtl_summary_dirs[(version, qtl_type)] / f"{tissue}.{suffix}.txt.gz"
    if not path.is_file():
        raise FileNotFoundError(f"Missing QTL summary: {path}")
    df = pd.read_csv(path, sep="\t", usecols=lambda c: c in {"gene_id", "phenotype_id", "qval", "pval_beta"})
    if "gene_id" not in df.columns:
        df["gene_id"] = df["phenotype_id"].str.split(":").str[-1]
    df = df[["gene_id", "qval", "pval_beta"]].dropna(subset=["qval", "pval_beta"])
    df["gene_id_stable"] = df["gene_id"].map(id_to_stable[version])
    df = df[df["gene_id_stable"].notna() & df["gene_id_stable"].isin(both_annotated_genes)]
    df = df.sort_values(["qval", "pval_beta"], kind="mergesort")
    return df.drop_duplicates("gene_id_stable", keep="first").set_index("gene_id_stable")[["qval", "pval_beta"]]


def storey_pi1(p, lambda_=0.5):
    """Storey π₁ = 1 − π₀ with fixed λ (same estimator TensorQTL/qvalue use under the hood)."""
    p = np.asarray(p, dtype=float)
    p = p[np.isfinite(p) & (p >= 0) & (p <= 1)]
    if len(p) < 10:
        return np.nan, len(p)
    pi0 = float(np.mean(p >= lambda_) / (1.0 - lambda_))
    pi0 = float(np.clip(pi0, 0.0, 1.0))
    return 1.0 - pi0, len(p)


def pi1_for_pair(disc, rep, gene_set=None, q_thresh=fdr_threshold):
    """π₁ among disc q < thresh genes that have a replication pval_beta."""
    hits = disc.index[disc["qval"] < q_thresh]
    if gene_set is not None:
        hits = hits.intersection(gene_set)
    p_rep = rep.reindex(hits)["pval_beta"].dropna()
    return storey_pi1(p_rep.to_numpy())


def genes_for_biotype(biotype):
    if biotype is None:
        return None  # overall = all both-annotated panel biotypes
    return set(stable_to_biotype.index[stable_to_biotype == biotype])


pi1_rows = []
biotype_levels = [None] + list(biotype_order)

for qtl_type in ("eqtl", "sqtl"):
    for tissue in tqdm(qtl_tissues, desc=f"pi1 {qtl_type}"):
        v10 = load_tissue_pval_qval(tissue, "v10", qtl_type)
        v11 = load_tissue_pval_qval(tissue, "v11", qtl_type)
        shared = v10.index.intersection(v11.index)
        v10s, v11s = v10.loc[shared], v11.loc[shared]
        for biotype in biotype_levels:
            gset = genes_for_biotype(biotype)
            label = "overall" if biotype is None else biotype
            for direction, disc, rep in (
                ("v10→v11", v10s, v11s),
                ("v11→v10", v11s, v10s),
            ):
                pi1, n = pi1_for_pair(disc, rep, gene_set=gset)
                pi1_rows.append(
                    {
                        "qtl_type": qtl_type,
                        "tissue": tissue,
                        "biotype": label,
                        "direction": direction,
                        "pi1": pi1,
                        "n_disc_with_rep_p": n,
                    }
                )

pi1_df = pd.DataFrame(pi1_rows)


### Storey π₁ replication by tissue (v10 ↔ v11)

In [ ]:
print(f"Storey π₁ rows: {len(pi1_df):,}; {pi1_df['tissue'].nunique()} tissues")


## Make figure


In [ ]:
thr05 = -np.log10(fdr_threshold)
thr01 = -np.log10(fdr_strong)
pad = display_max * 0.04
tick_vals = [0, 2, 4, 6, 8, display_max]
tick_labs = ["NT", "2", "4", "6", "8", f">{display_max:g}"]  # NT = not tested (q=1 → -log10q=0)
# strong discordance: q < 0.01 on one axis and q ≥ 0.05 on the other
shade_kwargs = dict(facecolor="0.88", edgecolor="none", alpha=0.9, zorder=0, lw=0)

fig, axes = plt.subplots(
    3, 3, figsize=(fig_width, fig_height), dpi=fig_dpi,
    sharex=False,
    sharey=False,
    constrained_layout=False,
)

# rows a / b: concordance scatters
for row, (qlabel, qtl_type) in enumerate([("eGenes", "eqtl"), ("sGenes", "sqtl")]):
    for col, biotype in enumerate(biotype_order):
        ax = axes[row, col]
        sub = pooled_calls[(qtl_type, biotype)]
        edge = gene_type_palette[biotype]
        x = sub["neglog10_q_v10"].clip(upper=display_max)
        y = sub["neglog10_q_v11"].clip(upper=display_max)
        n_strong = int(sub["strong_flip"].sum())
        ccc = pooled_ccc[(qtl_type, biotype)]

        ax.add_patch(Rectangle(
            (thr01, -pad),
            display_max + pad - thr01,
            thr05 + pad,
            **shade_kwargs,
        ))
        ax.add_patch(Rectangle(
            (-pad, thr01),
            thr05 + pad,
            display_max + pad - thr01,
            **shade_kwargs,
        ))

        ax.scatter(
            x, y,
            s=6, facecolors="none", edgecolors=edge,
            linewidths=0.35, alpha=0.45, rasterized=True, zorder=2,
        )
        ax.axhline(thr05, color="0.55", lw=0.6, ls="--", zorder=1)
        ax.axvline(thr05, color="0.55", lw=0.6, ls="--", zorder=1)
        ax.axhline(thr01, color="0.25", lw=0.7, ls="--", zorder=1)
        ax.axvline(thr01, color="0.25", lw=0.7, ls="--", zorder=1)
        ax.set_xlim(-pad, display_max + pad)
        ax.set_ylim(-pad, display_max + pad)
        ax.set_xticks(tick_vals)
        ax.set_yticks(tick_vals)
        ax.set_xticklabels(tick_labs)
        ax.set_yticklabels(tick_labs)
        ax.tick_params(labelbottom=True, labelleft=True, pad=1.5)
        # left-anchor square axes so ylabel sits next to ticks (not in empty padding)
        ax.set_aspect("equal", adjustable="box", anchor="W")
        ax.spines["right"].set_visible(False)
        ax.spines["top"].set_visible(False)

        ax.set_title(
            f"{biotype_labels[biotype]}\n(CCC={ccc:.3f}, {n_strong} strongly discordant)",
            fontsize=5.5,
        )
        ax.set_ylabel(r"$v_{11}$ $-\log_{10} q$", labelpad=2)
        ax.set_xlabel(r"$v_{10}$ $-\log_{10} q$", labelpad=2)

    axes[row, 0].annotate(
        qlabel,
        xy=(0, 0.5), xycoords="axes fraction",
        xytext=(-0.28, 0.5), textcoords="axes fraction",
        ha="center", va="center", rotation=90,
        annotation_clip=False, fontsize=7,
    )
    axes[row, 0].text(
        -0.28, 1.12, "ab"[row],
        transform=axes[row, 0].transAxes,
        va="top", ha="center", fontweight="bold", fontsize=9,
    )

# row c: ambiguity vs exon turnover, one panel per biotype
qtl_markers = {"eqtl": "o", "sqtl": "s"}
qtl_labels = {"eqtl": "eGene", "sqtl": "sGene"}

for col, biotype in enumerate(biotype_order):
    ax = axes[2, col]
    edge = gene_type_palette[biotype]
    for qtl_type, marker in qtl_markers.items():
        sub = pooled_calls[(qtl_type, biotype)]
        strong = sub.loc[
            sub["strong_flip"],
            ["delta_ambig", "pct_exon_turnover"],
        ].dropna()
        if strong.empty:
            continue
        ax.scatter(
            strong["pct_exon_turnover"],
            strong["delta_ambig"],
            s=10,
            marker=marker,
            facecolors="none",
            edgecolors=edge,
            linewidths=0.45,
            alpha=0.55,
            rasterized=True,
            zorder=2,
            label=qtl_labels[qtl_type],
        )

    ax.axhline(0, color="0.55", lw=0.5, ls="--", zorder=1)
    ax.set_xscale("symlog", linthresh=1)
    ax.set_xlim(-.5, None)
    ax.set_ylim(-105, 105)
    ax.spines["right"].set_visible(False)
    ax.spines["top"].set_visible(False)
    ax.tick_params(labelbottom=True, labelleft=True, pad=1.5)
    ax.set_title(biotype_labels[biotype], fontsize=5.5)
    ax.set_xlabel(r"exon bp unique to one release (% of $v_{47}$)", labelpad=2)
    ax.set_ylabel(r"$v_{47}-v_{39}$ % ambiguous", labelpad=2)
    if col == 0:
        ax.legend(loc="upper right", frameon=False, fontsize=4.5, handlelength=1.2)

axes[2, 0].text(
    -0.28, 1.12, "c",
    transform=axes[2, 0].transAxes,
    va="top", ha="center", fontweight="bold", fontsize=9,
)

fig.subplots_adjust(wspace=0.45, hspace=0.42, left=0.10, right=0.98, top=0.94, bottom=0.06)
plt.show()

## Save figure


In [ ]:
fig_dir.mkdir(parents=True, exist_ok=True)
fig.savefig(fig_pdf, bbox_inches="tight")
fig.savefig(fig_png, bbox_inches="tight")
print(f"Wrote {fig_pdf}")
print(f"Wrote {fig_png}")

## Print stats


In [ ]:
# Protein-coding CCC and strongly discordant genes (Results + caption a,b)
for qtl_type, qtl_label in (("eqtl", "eGene"), ("sqtl", "sGene")):
    sub = pooled_calls[(qtl_type, "protein_coding")]
    print(
        f"protein-coding {qtl_label}: CCC = {pooled_ccc[(qtl_type, 'protein_coding')]:.3f}; "
        f"strongly discordant = {int(sub['strong_flip'].sum())}"
    )

# Caption b: lncRNA / pseudogene sGene concordance
print(f"lncRNA sGene CCC = {pooled_ccc[('sqtl', 'lncRNA')]:.3f}")
print(f"lncRNA sGene strongly discordant = {int(pooled_calls[('sqtl', 'lncRNA')]['strong_flip'].sum())}")
print(f"pseudogene sGene CCC = {pooled_ccc[('sqtl', 'pseudogene')]:.3f}")

# Storey π₁ for protein-coding (Results)
pi1_mean = pi1_df.groupby(["qtl_type", "direction", "biotype"])["pi1"].mean()
print(
    f"Storey π₁ protein-coding eQTL (v10→v11): {pi1_mean[('eqtl', 'v10→v11', 'protein_coding')]:.3f}"
)
print(
    f"Storey π₁ protein-coding sQTL (v10→v11): {pi1_mean[('sqtl', 'v10→v11', 'protein_coding')]:.3f}"
)
